# MOD-12930 — balanced full suite

Same contenders and axes as the main (imbalanced) suite, but the text query of every
(size, depth) cell is **calibrated** so the SEARCH mirror's p50 matches the VSIM mirror's
(±20%, geometric bisection on the target match count; the achieved ratio is recorded).

**Why balance:** with skewed branches, `hybrid ≈ max + C` and `hybrid ≈ sum + C'` fit the
same measurement (sum − max = the small branch ≈ noise), so neither concurrency nor
overhead is identifiable. Equal branches maximize sum − max, so the workers-0/workers-N
pair separates "branches overlap" from "serial overhead C" and yields C twice, independently.

**Matrix:** size (10K/100K/500K) × depth K/W (10/20, 50/100, 250/500 — giving the
C(WINDOW) curve) × workers=2 × fields (none/title+text) × 4 contenders.
A cell whose calibration cannot reach balance is recorded with `balanced=False`.

In [1]:
import json, os
import pandas as pd

# Set REUSE_RESULTS=1 to re-render the tables from a previously saved run
# without re-executing the benchmark.
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_balanced_full.json'):
    d = json.load(open('results_balanced_full.json'))
    results, gates, meta = d['results'], d['gates'], d['meta']
    print('reusing saved results_balanced_full.json')
else:
    import bench_lib as B
    import balanced_lib as BAL
    titles, texts, emb, corpus_max = B.load_data()
    results, gates, meta = BAL.run_balanced_full(titles, texts, emb, corpus_max)
    with open('results_balanced_full.json', 'w') as f:
        json.dump(dict(meta=meta, results=results, gates=gates), f, indent=2, default=str)
    pd.DataFrame(results).to_csv('results_balanced_full.csv', index=False)
    print('saved results_balanced_full.json / .csv')

reusing saved results_balanced_full.json


## Calibration & gates

`balance_ratio` = search p50 / vsim p50 at calibration (1.0 = perfect balance; a cell counts as balanced within ±20%).

In [2]:
pd.DataFrame(gates)

,size,k,window,matches_mean,balance_ratio,gate_linear,gate_rrf
0,10000,10,20,2469.37500,1.17,16/16,16/16
1,10000,50,100,2469.37500,0.97,16/16,16/16
2,10000,250,500,3335.62500,1.16,16/16,16/16
3,100000,10,20,587.15625,0.81,16/16,16/16
4,100000,50,100,5626.56250,0.89,16/16,16/16
5,100000,250,500,14057.71875,1.13,16/16,16/16
6,500000,10,20,1336.15625,0.95,16/16,16/16
7,500000,50,100,4827.93750,1.05,16/16,16/16
8,500000,250,500,9446.31250,0.88,16/16,16/16


## p50 latency (ms)

In [3]:
df = pd.DataFrame(results)
pivot = df.pivot_table(index=['size', 'window', 'workers', 'fields'], columns='contender',
                       values='p50_ms', sort=False).round(2)
pivot[['hybrid_linear', 'hybrid_rrf', 'search_branch', 'vsim_branch']]

contender                         hybrid_linear  hybrid_rrf  search_branch  \
size   window workers fields                                                 
10000  20     2       none                 0.41        0.43           0.30   
                      title+text           0.48        0.48           0.38   
       100    2       none                 0.76        0.77           0.38   
                      title+text           0.88        0.87           0.50   
       500    2       none                 2.42        2.43           1.05   
                      title+text           2.75        2.73           1.11   
100000 20     2       none                 0.32        0.31           0.19   
                      title+text           0.36        0.38           0.22   
       100    2       none                 0.87        0.87           0.46   
                      title+text           1.01        1.04           0.53   
       500    2       none                 3.56        3.54           1.89   
                      title+text           3.86        3.84           1.97   
500000 20     2       none                 1.49        1.44           0.94   
                      title+text           1.63        1.59           1.22   
       100    2       none                 4.21        4.25           2.78   
                      title+text           4.60        4.67           3.40   
       500    2       none                12.85       12.89           6.41   
                      title+text          14.80       14.71           6.93   

contender                         vsim_branch  
size   window workers fields                   
10000  20     2       none               0.16  
                      title+text         0.25  
       100    2       none               0.34  
                      title+text         0.35  
       500    2       none               0.88  
                      title+text         0.92  
100000 20     2       none               0.20  
                      title+text         0.27  
       100    2       none               0.43  
                      title+text         0.53  
       500    2       none               1.46  
                      title+text         1.52  
500000 20     2       none               1.10  
                      title+text         1.19  
       100    2       none               1.68  
                      title+text         2.31  
       500    2       none               5.45  
                      title+text         5.70

## C — the serial overhead, and the C(WINDOW) curve

fields=none. `C_w0 = hybrid − (search+vsim)`, `C_w6 = hybrid − max(search,vsim)`
(mean latency). If depletion truly overlaps and C is genuinely serial, the two agree.
`C/max` says how the overhead compares to running one entire branch.

In [4]:
m = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'workers'],
        columns='contender', values='mean_ms', sort=False)
mx = m[['search_branch', 'vsim_branch']].max(axis=1)
sm = m[['search_branch', 'vsim_branch']].sum(axis=1)
c = pd.DataFrame({'hybrid': m['hybrid_linear'], 'max_branch': mx, 'sum_branch': sm})
c['C_ms'] = (c['hybrid'] - c['sum_branch']).where(
    c.index.get_level_values('workers') == 0, c['hybrid'] - c['max_branch'])
c['C_pct_of_max'] = (100 * c['C_ms'] / c['max_branch'])
c['gain_hint'] = None
c.round(2)

hybrid  max_branch  sum_branch  C_ms  C_pct_of_max  \
size   window workers                                                       
10000  20     2          0.42        0.32        0.48  0.09         28.80   
       100    2          0.79        0.43        0.75  0.37         85.79   
       500    2          2.46        1.08        1.97  1.38        127.35   
100000 20     2          0.32        0.20        0.39  0.12         58.91   
       100    2          0.94        0.55        0.98  0.40         72.54   
       500    2          3.65        1.68        3.13  1.96        116.64   
500000 20     2          1.76        1.24        2.32  0.53         42.56   
       100    2          4.95        3.22        4.99  1.74         53.94   
       500    2         14.09        6.96       12.72  7.13        102.40   

                      gain_hint  
size   window workers            
10000  20     2            None  
       100    2            None  
       500    2            None  
100000 20     2            None  
       100    2            None  
       500    2            None  
500000 20     2            None  
       100    2            None  
       500    2            None

### Parallelism gain per depth (`p50(w0)/p50(w_max)`, fields=none) — only when both were measured

In [5]:
ws = sorted(df['workers'].unique())
if len(ws) > 1 and 0 in ws:
    p = df[df.fields == 'none'].pivot_table(index=['size', 'window', 'contender'],
            columns='workers', values='p50_ms', sort=False)
    p.columns = [f'w{c}' for c in p.columns]
    p['gain'] = (p['w0'] / p[f'w{max(ws)}']).round(2)
    display(p.round(2))
else:
    print(f'single workers setting measured ({ws}) — gain not applicable')

single workers setting measured ([np.int64(2)]) — gain not applicable


## Merger scenario sweep

C = hybrid − max(search, vsim) with window, size and selectivity varied **independently**
(the calibrated matrix above ties |matches| to the vector branch's latency). Grid:
size × window × match-fraction (1% / 10% / 50% of corpus), fields=none.

In [6]:
if os.environ.get('REUSE_RESULTS') and os.path.exists('results_merger_sweep.json'):
    sw = json.load(open('results_merger_sweep.json'))
    sweep_results, sweep_meta = sw['results'], sw['meta']
    print('reusing saved results_merger_sweep.json')
else:
    import bench_lib as B
    from merger_sweep import run_sweep
    sweep_results, sweep_meta = run_sweep(*B.load_data())

sdf = pd.DataFrame(sweep_results)
sdf.pivot_table(index='window', columns=['size', 'match_frac'], values='C_ms').round(2)


===== merger sweep, dataset size 10000 =====


loaded+indexed 10000 docs in 15.5s (hash_indexing_failures=0)


target 100 (1%) -> measured |matches|≈187


n=10000 frac=1% k/w=10/20 hybrid_linear  p50=1.48ms


n=10000 frac=1% k/w=10/20 search_branch  p50=1.15ms


...

n=500000 frac=50% k/w=50/100 vsim_branch    p50=0.48ms
  -> C = 5.23ms


n=500000 frac=50% k/w=250/500 hybrid_linear  p50=40.87ms


n=500000 frac=50% k/w=250/500 search_branch  p50=37.35ms


n=500000 frac=50% k/w=250/500 vsim_branch    p50=1.78ms
  -> C = 3.74ms
saved results_merger_sweep.json


size       10000              100000              500000            
match_frac   0.01  0.10  0.50   0.01  0.10   0.50   0.01  0.10  0.50
window                                                              
20           0.49 -0.04  0.88   0.78  1.04   2.18   0.30  1.32  3.08
100          0.69  1.60  2.30   1.65  2.59   2.06   0.64  2.10  5.23
500          2.64  4.00  5.75   4.20  6.77 -82.50   1.84  4.02  3.74